# Google Analytics API Exploration
This script connects to the Gooogle Analytics API to pull Exploration data.

**This script is current still in development.**

You will need the following to use this script:
- [Google Application Credential from Google Cloud](https://docs.cloud.google.com/docs/authentication/provide-credentials-adc)
- [GA4 Property ID](https://support.google.com/analytics/answer/9539598?hl=en)

In [ ]:
# install the google-analytics-data module
!pip install google-analytics-data

import os
import pandas as pd
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import DateRange, Dimension, Metric, RunReportRequest, OrderBy, Filter, FilterExpression, CohortSpec

In [ ]:
# 1. Point this to the absolute path of your downloaded Service Account JSON file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = ""

# 2. Replace this with your actual GA4 Property ID (Numbers only, e.g., 123456789)
GA4_PROPERTY_ID = ""

def run_test_report():
    try:
        # Initialize the client (automatically reads the environment variable above)
        client = BetaAnalyticsDataClient()

        # Construct a simple flat-table request to test the connection
        request = RunReportRequest(
            property=f"properties/{GA4_PROPERTY_ID}",
            dimensions=[
                Dimension(name="pagePath"),
                Dimension(name="sessionSourceMedium")
            ],
            metrics=[
                Metric(name="activeUsers"),
                Metric(name="conversions")
            ],
            date_ranges=[
                DateRange(start_date="7daysAgo", end_date="today")
            ],
            limit=10  # Pull a small sample first
        )

        print("Sending request to Google Analytics Data API...")
        response = client.run_report(request)

        # Print column headers
        headers = [d.name for d in request.dimensions] + [m.name for m in request.metrics]
        print(f"\nSUCCESS! Found {len(response.rows)} rows.")
        print("-" * 60)
        print(" | ".join(headers))
        print("-" * 60)

        # Print row data
        for row in response.rows:
            dimensions = [val.value for val in row.dimension_values]
            metrics = [val.value for val in row.metric_values]
            print(" | ".join(dimensions + metrics))

    except Exception as e:
        print(f"\nERROR: Authentication or request failed.")
        print(f"Details: {e}")

if __name__ == "__main__":
    run_test_report()

Sending request to Google Analytics Data API...

SUCCESS! Found 10 rows.
------------------------------------------------------------
pagePath | sessionSourceMedium | activeUsers | conversions
------------------------------------------------------------
/ | (direct) / (none) | 5077 | 3
/portal/login.html | google / organic | 4849 | 0
/portal/login.html | (direct) / (none) | 3801 | 2
/ | google / organic | 3729 | 0
/graduate/ | fbinsta / socialmedia | 1574 | 0
/news/news-listing.html | (direct) / (none) | 1093 | 0
/graduate/ | digital / adtaxi | 1040 | 0
/search/ | google / organic | 932 | 0
/portal/login.html | api-2b76e504.duosecurity.com / referral | 899 | 0
/search/ | (direct) / (none) | 796 | 0


### How to replicate your GA4 Exploration in code:

1.  **Open your Exploration:** Navigate to your Google Analytics 4 property, then go to `Explorations` and open the specific exploration you want to replicate.
2.  **Identify Dimensions and Metrics:** Look at the `Dimensions` and `Metrics` sections. Note down the exact names of all dimensions and metrics you are using in your exploration. For example, `Page path`, `Event name`, `Active users`, `Conversions`.
3.  **Identify Filters/Segments:** Check the `Filters` and `Segments` sections. These are crucial for narrowing down your data. Note the dimension/metric being filtered, the operator (e.g., `exactly matches`, `contains`), and the value.
4.  **Date Range:** Note the date range applied to your exploration.
5.  **Reconstruct the Request:** Use the collected information to populate the `dimensions`, `metrics`, `date_ranges`, and `dimension_expression_filters` (or `metric_expression_filters`) in the `RunReportRequest`.

In [ ]:
# Define my array of page paths
degree_paths = [
    "/sciences-math/mathematics.html",
    "graduate/socialwork.html",
    "/business/accounting.html",
    "health-sciences/nursing.html",
    "arts-humanities/africana-studies.html",
    "arts-humanities/performing-arts.html",
    "sciences-math/physics.html",
    "/arts-humanities/philosophy-religion.html",
    "/sciences-math/biochemistry.html",
    "/social-behavioral-sciences/political-science.html",
    "/sciences-math/biology.html",
    "/social-behavioral-sciences/psychology.html",
    "/business/business-analytics.html",
    "/health-sciences/public-health.html",
    "/business/business-studies-program.html",
    "/social-behavioral-sciences/social-work-bsw/index.html",
    "/sciences-math/chemistry.html",
    "/social-behavioral-sciences/sociology-anthropology.html",
    "/arts-humanities/communication-studies.html",
    "/arts-humanities/academic-programs.html",
    "/business/computer-information-systems.html",
    "/sciences-math/sustainability.html",
    "/business/computer-science.html",
    "/education/education-programs.html",
    "/social-behavioral-sciences/criminal-justice.html",
    "/graduate/accounting.html",
    "/arts-humanities/digital-studies.html",
    "/graduate/american-studies.html",
    "/social-behavioral-sciences/economics.html",
    "/graduate/business-administration.html",
    "/business/entrepreneurship.html",
    "/graduate/coastal-zone-management.html",
    "/sciences-math/environmental-science.html",
    "/graduate/counseling.html",
    "/business/esports-management.html",
    "/graduate/data-science_strategic-analytics.html",
    "/health-sciences/exercise-science.html",
    "/graduate/education.html",
    "/business/finance.html",
    "/graduate/environmental-science.html",
    "/sciences-math/geology.html",
    "/graduate/mba-healthcare-administration.html",
    "/health-sciences/health-science.html",
    "/graduate/holocaust-genocide-studies.html",
    "/business/hemp-cannabis-business-management.html",
    "/graduate/instructional-technology.html",
    "/arts-humanities/historical-studies.html",
    "/graduate/nursing.html",
    "/sciences-math/energy.html",
    "/business/hospitality-tourism-program.html",
    "/graduate/doctor_nursing_practice.html",
    "/arts-humanities/languages-culture.html",
    "/arts-humanities/literature.html",
    "/graduate/physical-therapy.html",
    "/sciences-math/marine-science.html",
    "/graduate/public-health.html",
    "/sciences-math/computer-science.html",
    "/business/business-administration.html"
]

def run_exploration_report(property_id):
    try:
        client = BetaAnalyticsDataClient()

        # Example: Replicating an Exploration with specific dimensions, metrics, and a filter
        # Customize these based on your GA4 Exploration settings.
        request = RunReportRequest(
            property=f"properties/{property_id}",
            dimensions=[
                Dimension(name="pagePathPlusQueryString"),
                Dimension(name="landingPagePlusQueryString"),
                Dimension(name="hour")
            ],
            metrics=[
                Metric(name="activeUsers"),
                Metric(name="totalUsers")
            ],
            date_ranges=[
                DateRange(start_date="2026-03-28", end_date="2026-05-30")
            ],
            dimension_filter=FilterExpression(
                filter=Filter(
                    field_name="pagePathPlusQueryString",
                    in_list_filter=Filter.InListFilter(
                      values=degree_paths
                    )
                )
            ),
            limit=100
        )

        print(f"Sending request to Google Analytics Data API for property {property_id}...")
        response = client.run_report(request)

        headers = [
            d.name for d in request.dimensions
        ] + [m.name for m in request.metrics]

        print(f"\nSUCCESS! Found {len(response.rows)} rows.")
        print("-" * (10 * len(headers)))
        print(" | ".join(headers))
        print("-" * (10 * len(headers)))

        data_rows = []
        for row in response.rows:
            dimensions = [val.value for val in row.dimension_values]
            metrics = [val.value for val in row.metric_values]
            data_rows.append(dimensions + metrics)

        df = pd.DataFrame(data_rows, columns=headers)
        print(df.to_string())
        return df

    except Exception as e:
        print(f"\nERROR: Request failed.")
        print(f"Details: {e}")
        return pd.DataFrame() # Return an empty DataFrame on error

# To run this function, call it with your GA4 Property ID
# For example:
report_df = run_exploration_report(GA4_PROPERTY_ID)


Sending request to Google Analytics Data API for property 267508725...

SUCCESS! Found 100 rows.
--------------------------------------------------
pagePathPlusQueryString | landingPagePlusQueryString | hour | activeUsers | totalUsers
--------------------------------------------------
                            pagePathPlusQueryString                       landingPagePlusQueryString hour activeUsers totalUsers
0         /graduate/holocaust-genocide-studies.html        /graduate/holocaust-genocide-studies.html   10         398        402
1         /graduate/holocaust-genocide-studies.html        /graduate/holocaust-genocide-studies.html   14         373        389
2         /graduate/holocaust-genocide-studies.html        /graduate/holocaust-genocide-studies.html   12         372        380
3         /graduate/holocaust-genocide-studies.html        /graduate/holocaust-genocide-studies.html   11         366        375
4         /graduate/holocaust-genocide-studies.html        /graduate/

In [ ]:
report_df['totalUsers'] = pd.to_numeric(report_df['totalUsers'])

# Pivot the DataFrame to get unique pagePathPlusQueryString as columns with totalUsers as values
pivoted_df = report_df.pivot_table(
    index=['landingPagePlusQueryString', 'hour'],
    columns='pagePathPlusQueryString',
    values='totalUsers',
    aggfunc='max'  # Changed from 'sum' to 'max'
).reset_index()

# Display the first few rows of the pivoted DataFrame
display(pivoted_df.head())

pagePathPlusQueryString,landingPagePlusQueryString,hour,/graduate/accounting.html,/graduate/american-studies.html,/graduate/business-administration.html,/graduate/education.html,/graduate/environmental-science.html,/graduate/holocaust-genocide-studies.html,/graduate/instructional-technology.html,/graduate/physical-therapy.html
0,/graduate/accounting.html,1,111.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,/graduate/accounting.html,10,159.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,/graduate/accounting.html,11,116.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,/graduate/accounting.html,12,113.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,/graduate/accounting.html,13,124.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The `run_exploration_report` function above provides a more detailed example of how to construct a `RunReportRequest` that includes filtering and ordering, common elements in GA4 Explorations. You will need to:

1.  **Modify the `dimensions` and `metrics` lists** to match those in your Google Analytics Exploration.
2.  **Adjust the `date_ranges`** to reflect the period you're interested in.
3.  **Update the `dimension_filter`** (and potentially add `metric_filter`) to replicate any segments or filters from your Exploration. If your exploration has multiple filters, you can combine them using `and_group` or `or_group` within `FilterExpression`.
4.  **Add `order_bys`** if you have specific sorting applied in your Exploration.
5.  **Uncomment and call `run_exploration_report(GA4_PROPERTY_ID)`** once you have customized the request.

In [ ]:
# Define the exact columns from the user's desired output
desired_path_columns = [
    '/health-sciences/nursing.html',
    '/graduate/socialwork.html',
    '/business/accounting.html',
    '/sciences-math/mathematics.html',
    '/sciences-math/physics.html',
    '/arts-humanities/performing-arts.html',
    '/arts-humanities/africana-studies.html'
]

# Ensure 'hour' is an integer for cleaner representation
pivoted_df['hour'] = pd.to_numeric(pivoted_df['hour'], errors='coerce').astype(int)

# Rename the 'landingPagePlusQueryString' column to 'landing_page' in the pivoted_df itself
pivoted_df = pivoted_df.rename(columns={'landingPagePlusQueryString': 'landing_page'})

# Define the full list of columns for the final DataFrame
# This includes 'landing_page', 'hour', and all desired_path_columns
all_desired_columns = ['landing_page', 'hour'] + desired_path_columns

# Reindex the pivoted_df to ensure all desired columns are present.
# Any column in all_desired_columns not in pivoted_df will be added as NaN.
# Any column in pivoted_df not in all_desired_columns will be dropped.
final_df = pivoted_df.reindex(columns=all_desired_columns)

# Fill any NaN values (either from reindexing missing columns or from the pivot itself) with 0
final_df = final_df.fillna(0)

display(final_df.head(8))

pagePathPlusQueryString,landing_page,hour,/health-sciences/nursing.html,/graduate/socialwork.html,/business/accounting.html,/sciences-math/mathematics.html,/sciences-math/physics.html,/arts-humanities/performing-arts.html,/arts-humanities/africana-studies.html
0,/graduate/accounting.html,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/graduate/accounting.html,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/graduate/accounting.html,11,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,/graduate/accounting.html,12,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,/graduate/accounting.html,13,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,/graduate/accounting.html,14,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,/graduate/accounting.html,15,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,/graduate/accounting.html,16,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
pivoted_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   landing_page                               100 non-null    object 
 1   hour                                       100 non-null    int64  
 2   /graduate/accounting.html                  20 non-null     float64
 3   /graduate/american-studies.html            9 non-null      float64
 4   /graduate/business-administration.html     9 non-null      float64
 5   /graduate/education.html                   11 non-null     float64
 6   /graduate/environmental-science.html       6 non-null      float64
 7   /graduate/holocaust-genocide-studies.html  24 non-null     float64
 8   /graduate/instructional-technology.html    4 non-null      float64
 9   /graduate/physical-therapy.html            17 non-null     float64
dtypes: float64(8), int64(1), ob

In [ ]:
pivoted_df['/graduate/accounting.html'].unique()

array([111., 159., 116., 113., 124., 112., 184., 107., 134., 122., 131.,
       119., 127., 136., 135.,  nan])